# Virtual-experiment thermodynamic diagnostics example

This notebook is a researcher-facing exploratory example for inspecting the standard `thermodynamic_diagnostics.csv` table and `DegradationScreenResult.thermodynamic_diagnostics()` accessor. It is not an empirical validation, calibration, literature comparison, or biology-evidence notebook.

The first run shows the normal header-only case when a virtual-experiment sample has no configured thermodynamic artifacts. The second run copies package-generated configured thermodynamic summary artifacts into the virtual-experiment sample bundle, then lets the standard table writer expose them. The virtual-experiment path does not infer activities, reaction quotients, concentrations, redox potentials, electron balances, validation evidence, entropy rates, or solver-time thermodynamic enforcement.


In [ ]:
import csv
import json
import os
import shutil
import sys
from pathlib import Path

import yaml

ROOT = Path.cwd()
SRC = ROOT / "src"
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from fungal_model import VirtualExperiment, run_configured_model

REGISTRY = ROOT / "data_registry" / "registry_index.yml"
SOURCE_CONFIG = ROOT / "data" / "model_configs" / "toy_homogeneous_ab.yml"
OUTPUT_ROOT = Path(os.environ.get("FUNGMOD_NOTEBOOK_OUTPUT_ROOT", str(ROOT / "notebooks" / "examples" / "Outputs")))
OUTPUT = OUTPUT_ROOT / "16_thermodynamic_diagnostics_example"
NO_ARTIFACT_OUTPUT = OUTPUT / "no_artifacts"
CONFIGURED_OUTPUT = OUTPUT / "configured_thermodynamic_summary"
COPIED_OUTPUT = OUTPUT / "copied_artifact_bridge"
CONFIG = OUTPUT / "configured_inputs" / "explicit_q_entropy_summary.yml"


## Header-only standard table

A normal virtual experiment does not automatically have configured thermodynamic summary artifacts in its sample bundles. In that case `thermodynamic_diagnostics.csv` is still part of the standard output schema, but it remains header-only. Empty rows mean "no configured artifacts were present," not "thermodynamics passed" or "thermodynamics failed.


In [ ]:
study = VirtualExperiment.from_registry(
    fungi=["sabiork_beta_glucosidase_source"],
    substrates=["cellobiose"],
    environments=["sabiork_reaction_618_selected_conditions"],
    registry=REGISTRY,
)

header_only_result = study.simulate(
    mode="exploratory",
    n_samples=1,
    seed=16,
    output_dir=NO_ARTIFACT_OUTPUT,
    quicklook=False,
)
header_only_rows = header_only_result.thermodynamic_diagnostics()
with (NO_ARTIFACT_OUTPUT / "thermodynamic_diagnostics.csv").open(newline="", encoding="utf-8") as handle:
    header_only_columns = next(csv.reader(handle))

{
    "thermodynamic_diagnostics_rows": len(header_only_rows),
    "has_standard_table": (NO_ARTIFACT_OUTPUT / "thermodynamic_diagnostics.csv").exists(),
    "selected_columns": [
        column
        for column in header_only_columns
        if column in {"row_name", "allowed_use", "interpretation_guardrail"}
    ],
}


## Package-generated configured artifacts

Create a small configured-run fixture with explicit thermodynamic metadata and let `run_configured_model(...)` generate `thermodynamic_summary.json` and `thermodynamic_summary.csv`. The configured metadata values are synthetic software-test fixture values with no biological claim; the important boundary is that the package creates the summary artifacts, and the notebook does not implement thermodynamic equations or hidden rate laws.


In [ ]:
OUTPUT.mkdir(parents=True, exist_ok=True)
CONFIG.parent.mkdir(parents=True, exist_ok=True)

source = "Configured synthetic explicit-Q thermodynamics fixture; no scientific claim."
config = yaml.safe_load(SOURCE_CONFIG.read_text(encoding="utf-8"))
config["name"] = "explicit Q entropy diagnostics bridge benchmark"
config["mode"] = "toy"
config["maturity"] = "framework_benchmark"
config.setdefault("validators", []).append(
    {
        "id": "explicit_q_gibbs",
        "validator_type": "reaction_quotient_thermodynamic_metadata",
        "estimate": {
            "reaction_name": "configured synthetic condition-specific reaction",
            "source": source,
            "delta_gibbs": {
                "name": "configured standard delta G",
                "symbol": "dG_standard_configured",
                "value": -5.0,
                "units": "kilojoule / mole",
                "uncertainty": 0.0,
                "source": source,
                "confidence_level": "medium",
                "notes": "Synthetic standard Gibbs value for configured-output diagnostics only.",
                "measurement_method": "defined fixture value",
                "validity_range": "not a biological validity range",
            },
            "conditions": {
                "parameters": [
                    {
                        "name": "configured thermodynamic temperature",
                        "symbol": "T_configured",
                        "value": 298.15,
                        "units": "kelvin",
                        "uncertainty": 0.0,
                        "source": source,
                        "confidence_level": "medium",
                        "notes": "Synthetic condition value for configured-output diagnostics only.",
                        "measurement_method": "defined fixture value",
                        "validity_range": "not a biological validity range",
                    }
                ]
            },
        },
        "reaction_quotient": {
            "name": "configured dimensionless reaction quotient",
            "symbol": "Q_configured",
            "value": 1.0,
            "units": "dimensionless",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "medium",
            "notes": "Explicit caller-supplied Q for configured-output diagnostics only.",
            "measurement_method": "defined fixture value",
            "validity_range": "not a biological validity range",
        },
        "temperature": {
            "name": "configured dynamic thermodynamic temperature",
            "symbol": "T_configured_dynamic",
            "value": 298.15,
            "units": "kelvin",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "medium",
            "notes": "Explicit caller-supplied temperature for configured-output diagnostics only.",
            "measurement_method": "defined fixture value",
            "validity_range": "not a biological validity range",
        },
    }
)
config["validators"].append(
    {
        "id": "explicit_entropy_rate",
        "validator_type": "entropy_production_rate_metadata",
        "condition_specific_delta_gibbs": {
            "name": "configured condition-specific delta G",
            "symbol": "dG_entropy_rate_configured",
            "value": -10.0,
            "units": "kilojoule / mole",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "medium",
            "notes": "Synthetic delta G for configured entropy-rate diagnostics only.",
            "measurement_method": "defined fixture value",
            "validity_range": "not a biological validity range",
        },
        "reaction_extent_rate": {
            "name": "configured reaction extent rate",
            "symbol": "xi_dot_configured",
            "value": 2.0,
            "units": "millimole / second",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "medium",
            "notes": "Explicit caller-supplied extent rate for configured entropy-rate diagnostics only.",
            "measurement_method": "defined fixture value",
            "validity_range": "not a biological validity range",
        },
        "temperature": {
            "name": "configured entropy-rate temperature",
            "symbol": "T_entropy_rate_configured",
            "value": 298.15,
            "units": "kelvin",
            "uncertainty": 0.0,
            "source": source,
            "confidence_level": "medium",
            "notes": "Explicit caller-supplied temperature for configured entropy-rate diagnostics only.",
            "measurement_method": "defined fixture value",
            "validity_range": "not a biological validity range",
        },
    }
)

CONFIG.write_text(yaml.safe_dump(config, sort_keys=False), encoding="utf-8")
configured_result = run_configured_model(CONFIG, output_dir=CONFIGURED_OUTPUT)
configured_summary = json.loads((CONFIGURED_OUTPUT / "thermodynamic_summary.json").read_text(encoding="utf-8"))
with (CONFIGURED_OUTPUT / "thermodynamic_summary.csv").open(newline="", encoding="utf-8") as handle:
    configured_rows = list(csv.DictReader(handle))

{
    "configured_output_directory": str(CONFIGURED_OUTPUT),
    "validation_statuses": {
        row["name"]: row["status"]
        for row in configured_result.validation_report()
        if row["name"] in {"reaction_quotient_thermodynamic_feasibility", "entropy_production_rate_metadata"}
    },
    "summary_kind": configured_summary["kind"],
    "has_reaction_quotient_gibbs": configured_summary["has_reaction_quotient_gibbs"],
    "has_entropy_production_rate": configured_summary["has_entropy_production_rate"],
    "has_entropy_budget": configured_summary["has_entropy_budget"],
    "row_names": [row["name"] for row in configured_rows],
}


## Standard virtual-experiment bridge

Copy only those package-generated configured summary artifacts into the virtual-experiment sample directory, then call the standard table writer again. `DegradationScreenResult.thermodynamic_diagnostics()` reads `thermodynamic_diagnostics.csv`; it does not rerun the configured model, validate the biological case, or infer missing thermodynamic inputs.


In [ ]:
bridge_result = study.simulate(
    mode="exploratory",
    n_samples=1,
    seed=17,
    output_dir=COPIED_OUTPUT,
    quicklook=False,
)
sample_dir = Path(bridge_result.screen_result.case_results[0].samples[0].output_directory)
for filename in ("thermodynamic_summary.json", "thermodynamic_summary.csv"):
    shutil.copy2(CONFIGURED_OUTPUT / filename, sample_dir / filename)

bridge_result.write_tables()
diagnostic_rows = bridge_result.thermodynamic_diagnostics()
report_path = bridge_result.write_report(COPIED_OUTPUT / "report", include_html=True, include_index=True)
report_text = report_path.read_text(encoding="utf-8")
index_text = report_path.with_name("index.html").read_text(encoding="utf-8")

row_names = sorted({row["row_name"] for row in diagnostic_rows})
allowed_uses = sorted({row["allowed_use"] for row in diagnostic_rows})
artifact_flags = {
    "json_present": sorted({row["thermodynamic_summary_json_present"] for row in diagnostic_rows}),
    "csv_present": sorted({row["thermodynamic_summary_csv_present"] for row in diagnostic_rows}),
}

{
    "thermodynamic_diagnostics_rows": len(diagnostic_rows),
    "row_names": row_names,
    "allowed_uses": allowed_uses,
    "artifact_flags": artifact_flags,
    "entropy_budget_statuses": sorted({row["entropy_budget_status"] for row in diagnostic_rows}),
    "report_mentions_standard_rows": "Standard virtual-experiment rows from `thermodynamic_diagnostics.csv`" in report_text,
    "index_links_standard_table": "thermodynamic_diagnostics.csv" in index_text,
    "guardrail_excerpt": diagnostic_rows[0]["interpretation_guardrail"],
}
